# Stage 11: Advanced OLAP & Multidimensional Data Cube Analysis
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Unit 3 — Data Warehouse and OLAP Technology for Data Mining (Multidimensional Data Model, Data Cubes, Star Schema, OLAP Operations: Roll-Up, Drill-Down, Slice, Dice, Pivot, Data Generalization, Attribute-Oriented Induction, Efficient Cube Computation, Iceberg Cubes)  
**Data Source**: Relational Data Warehouse (`data/database/cybercrime.db`)  
**Sample Size**: $N = 36$ State/UT Observations, 40 Leaf Offense Categories, 18 Specific Motives, 5 Historical Trend Years (2018–2022)  

---

## 1. Objective & Methodological Scope

### Purpose:
This stage implements a formal **Multidimensional Data Cube and Advanced OLAP Analysis Layer** extending the foundational warehouse created in Stage 3. It establishes the full cuboid lattice, executes core OLAP navigation operations, demonstrates semantic data generalization via Attribute-Oriented Induction (AOI), and illustrates selective cuboid materialization through Iceberg cube computation.

### Key Curriculum Concepts Covered:
1. **Multidimensional Data Modeling**: Formal grain audit separating Category, Motive, and Historical Trend dimensions to prevent invalid Cartesian products.
2. **Lattice of Cuboids**: Construction of base cuboids ($0\text{-D}$ to $3\text{-D}$) spanning `State/UT`, `Act Group`, `Crime Category`, and `National Total`.
3. **Core OLAP Operations**: Programmatic execution of **Roll-Up**, **Drill-Down**, **Slice**, **Dice**, and **Pivot**.
4. **Data Generalization & Attribute-Oriented Induction (AOI)**: Transforming granular state-category facts into high-level concept tuples, achieving $99.58\%$ tuple count reduction.
5. **Efficient Cube Computation & Iceberg Cuboids**: Demonstration of selective materialization filtering on high-volume analytical thresholds ($\ge 1,000$ cases).

### Critical Methodological Guardrails:
- **Strictly Descriptive & Non-Causal**: Aggregations summarize recorded volumes; they do not infer criminological causes.
- **Fact Grain Separation**: Category facts ($1,764$ rows) and Motive facts ($684$ rows) operate at distinct dimensional grains and are never cross-joined blindly.


In [1]:
# Setup paths and imports
import os
import sys
import sqlite3
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 11 advanced OLAP module
from src.advanced_olap import (
    get_db_connection,
    compute_cuboids,
    compute_state_act_group_pivot,
    execute_slice_operation,
    execute_dice_operation,
    execute_drill_down_operation,
    perform_attribute_oriented_induction,
    compute_iceberg_cuboid,
    run_stage11_pipeline
)

print("Stage 11 advanced OLAP and data cube routines loaded.")

Stage 11 advanced OLAP and data cube routines loaded.


## 2. Warehouse Fact-Grain Audit & Dimensional Separation

Before constructing cuboids, we audit the dimensional grain of the relational warehouse tables in `data/database/cybercrime.db`.

| Fact Table | Dimensional Grain | Total Rows | Measure | Reconciled Total |
|---|---|---|---|---|
| `fact_cybercrime_category_2023` | $\text{State} \times \text{Category} \times \text{Year (2023)}$ | $1,764$ ($36 \times 49$) | `cases` | **$86,420$** (Leaf Sum) |
| `fact_cybercrime_motive_2023` | $\text{State} \times \text{Motive} \times \text{Year (2023)}$ | $684$ ($36 \times 19$) | `motive_count` | **$86,420$** (Specific Sum) |
| `fact_cybercrime_trend` | $\text{State} \times \text{Year (2018–2022)}$ | $180$ ($36 \times 5$) | `cases` | Longitudinal Panel |

> **Critical Grain Rule**: Because NCRB does not provide cross-tabulated Category $\times$ Motive microdata, we construct separate compatible cubes: $\text{Cube}_{\text{Category}}$, $\text{Cube}_{\text{Motive}}$, and $\text{Cube}_{\text{Trend}}$.

In [2]:
conn = get_db_connection()
tables = [t[0] for t in conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';").fetchall()]
table_counts = {t: conn.execute(f"SELECT count(*) FROM {t};").fetchone()[0] for t in tables}
print("=== WAREHOUSE TABLE ROW COUNTS ===")
for t, cnt in table_counts.items():
    print(f"  - {t:<32}: {cnt:>5} rows")
conn.close()

=== WAREHOUSE TABLE ROW COUNTS ===
  - dim_state                       :    36 rows
  - dim_year                        :     6 rows
  - dim_crime_category              :    49 rows
  - dim_motive                      :    19 rows
  - fact_cybercrime_category_2023   :  1764 rows
  - fact_cybercrime_motive_2023     :   684 rows
  - fact_cybercrime_trend           :   180 rows


## 3. Base Cuboids & Lattice Computation

We extract the base cuboid ($	ext{State} \times \text{Leaf Category}$, $1,440$ tuples) and pre-materialize higher-level roll-up cuboids.

In [3]:
cuboids = compute_cuboids()
print("=== BASE CUBOID: STATE x LEAF CATEGORY (First 8 Rows of 1,440) ===")
display(cuboids["state_category"].head(8))

print("\n=== ROLL-UP CUBOID: NATIONAL x ACT GROUP (3 Tuples) ===")
display(cuboids["national_act_group"])

=== BASE CUBOID: STATE x LEAF CATEGORY (First 8 Rows of 1,440) ===

=== ROLL-UP CUBOID: NATIONAL x ACT GROUP (3 Tuples) ===


,state_id,state_name,is_ut,admin_type,category_id,category_display_name,act_group,parent_category,cases
0,1,Andaman and Nicobar Islands,1,Union Territory,1,Tampering Computer Source documents (Sec.65),IT Act,Offences under I.T. Act,0
1,1,Andaman and Nicobar Islands,1,Union Territory,4,Computer Related Offences - Computer Related Offences (Sec.66) - a1) Ransom-ware,IT Act,Offences under I.T. Act,0
2,1,Andaman and Nicobar Islands,1,Union Territory,5,Computer Related Offences - Computer Related Offences (Sec.66) - a2) Offences other than Ransom-ware,IT Act,Offences under I.T. Act,0
3,1,Andaman and Nicobar Islands,1,Union Territory,6,Computer Related Offences - B) Dishonesty receiving stolen computer resource or communication device (Sec.66B),IT Act,Offences under I.T. Act,0
4,1,Andaman and Nicobar Islands,1,Union Territory,7,Computer Related Offences - C) Identity Theft (Sec.66C),IT Act,Offences under I.T. Act,1
5,1,Andaman and Nicobar Islands,1,Union Territory,8,Computer Related Offences - D) Cheating by personation by using computer resource (Sec.66D),IT Act,Offences under I.T. Act,0
6,1,Andaman and Nicobar Islands,1,Union Territory,9,Computer Related Offences - E) Violation of Privacy (Sec.66E),IT Act,Offences under I.T. Act,0
7,1,Andaman and Nicobar Islands,1,Union Territory,10,Cyber Terrorism (Sec.66 F),IT Act,Offences under I.T. Act,0


,act_group,leaf_categories_count,total_cases,national_share_pct
0,IT Act,18,44237,51.19
1,IPC,17,41849,48.43
2,SLL,5,334,0.39


## 4. OLAP Operations: Roll-Up & Drill-Down

### 4.1 Roll-Up
Aggregating from 40 specific leaf offenses $\rightarrow$ 3 Act Groups $\rightarrow$ National Grand Total ($86,420$ cases):
- **IT Act**: $44,237$ cases ($51.19\%$)
- **IPC Crimes r/w IT Act**: $41,849$ cases ($48.43\%$)
- **SLL Crimes r/w IT Act**: $334$ cases ($0.39\%$)

In [4]:
print("=== ROLL-UP CUBOID: ADMIN TYPE x ACT GROUP (6 Tuples) ===")
display(cuboids["admin_act_group"])

=== ROLL-UP CUBOID: ADMIN TYPE x ACT GROUP (6 Tuples) ===


,admin_type,act_group,jurisdictions_count,total_cases,national_share_pct
0,State (28),IT Act,28,43739,50.61
1,State (28),IPC,28,41539,48.07
2,State (28),SLL,28,325,0.38
3,Union Territory (8),IT Act,8,498,0.58
4,Union Territory (8),IPC,8,310,0.36
5,Union Territory (8),SLL,8,9,0.01


### 4.2 Drill-Down
Decomposing the highest volume IT Act offense (**Section 66D Personation**, $24,028$ cases) down to individual State/UT jurisdictions.

In [5]:
drill_df = execute_drill_down_operation()
print("=== DRILL-DOWN: SECTION 66D CHEATING BY PERSONATION (Top 8 States) ===")
display(drill_df.head(8))

=== DRILL-DOWN: SECTION 66D CHEATING BY PERSONATION (Top 8 States) ===


,state_name,is_ut,act_group,category_display_name,section_reference,state_category_cases,pct_of_sec66d_national


## 5. OLAP Operations: Slice, Dice & Pivot

### 5.1 Slice (Single-Dimension Filtering)
Isolating the `IT Act` slice across all 36 jurisdictions ($44,237$ total cases).

In [6]:
slice_df = execute_slice_operation(act_group_filter="IT Act")
print("=== SLICE: ACT GROUP = 'IT Act' (Top 8 States) ===")
display(slice_df.head(8))

=== SLICE: ACT GROUP = 'IT Act' (Top 8 States) ===


,state_name,is_ut,act_group,sliced_cases,pct_of_act_group_total
0,Karnataka,0,IT Act,21870,49.44
1,Uttar Pradesh,0,IT Act,10102,22.84
2,Tamil Nadu,0,IT Act,1912,4.32
3,Rajasthan,0,IT Act,1464,3.31
4,Kerala,0,IT Act,1089,2.46
5,Jharkhand,0,IT Act,993,2.24
6,Maharashtra,0,IT Act,960,2.17
7,Odisha,0,IT Act,892,2.02


### 5.2 Dice (Multi-Dimensional Sub-Cube Extraction)
Extracting a sub-cube defined by $\{\text{Top 5 Volume States}\} \times \{\text{IT Act}, \text{IPC}\}$.

In [7]:
dice_df = execute_dice_operation(top_n_states=5)
print("=== DICE: TOP 5 STATES x {IT Act, IPC} (10 Sub-Cube Cells) ===")
display(dice_df)

=== DICE: TOP 5 STATES x {IT Act, IPC} (10 Sub-Cube Cells) ===


,state_name,act_group,leaf_categories_count,dicing_cases
0,Bihar,IPC,17,4052
1,Bihar,IT Act,18,397
2,Karnataka,IPC,17,18
3,Karnataka,IT Act,18,21870
4,Maharashtra,IPC,17,7131
5,Maharashtra,IT Act,18,960
6,Telangana,IPC,17,17787
7,Telangana,IT Act,18,439
8,Uttar Pradesh,IPC,17,689
9,Uttar Pradesh,IT Act,18,10102


### 5.3 Pivot (Cross-Tabulation Matrix)
Reorienting dimensions: Rows = State/UT, Columns = Act Groups, Values = Cases & Shares.

In [8]:
pivot_df = compute_state_act_group_pivot()
print("=== PIVOT TABLE: STATE x ACT GROUP (Top 10 States by Volume) ===")
display(pivot_df.head(10))

=== PIVOT TABLE: STATE x ACT GROUP (Top 10 States by Volume) ===


,state_id,state_name,is_ut,admin_type,it_act_cases,ipc_cases,sll_cases,total_leaf_cases,it_act_share_pct,ipc_share_pct,sll_share_pct
0,16,Karnataka,0,State,21870,18,1,21889,99.91,0.08,0.00
1,32,Telangana,0,State,439,17787,10,18236,2.41,97.54,0.05
2,34,Uttar Pradesh,0,State,10102,689,3,10794,93.59,6.38,0.03
3,21,Maharashtra,0,State,960,7131,12,8103,11.85,88.00,0.15
4,5,Bihar,0,State,397,4052,1,4450,8.92,91.06,0.02
5,31,Tamil Nadu,0,State,1912,2195,14,4121,46.40,53.26,0.34
6,17,Kerala,0,State,1089,2110,96,3295,33.05,64.04,2.91
7,29,Rajasthan,0,State,1464,953,18,2435,60.12,39.14,0.74
8,26,Odisha,0,State,892,1393,63,2348,37.99,59.33,2.68
9,2,Andhra Pradesh,0,State,362,1978,1,2341,15.46,84.49,0.04


## 6. Data Generalization & Attribute-Oriented Induction (AOI)

Attribute-Oriented Induction (AOI) compresses low-level attribute values into generalized concept descriptions using concept hierarchies:
- $\text{State/UT} \rightarrow \text{Administrative Type (State vs Union Territory)}$
- $\text{Crime Category (40 leaves)} \rightarrow \text{Statutory Act Group (IT Act, IPC, SLL)}$

**Result**: Compresses $1,440$ base tuples into **6 generalized concept tuples** ($99.58\%$ reduction in tuple cardinality).

In [9]:
aoi_df = perform_attribute_oriented_induction()
print("=== ATTRIBUTE-ORIENTED INDUCTION (AOI): GENERALIZED RELATION ===")
display(aoi_df)

=== ATTRIBUTE-ORIENTED INDUCTION (AOI): GENERALIZED RELATION ===


,generalized_admin_type,generalized_act_group,generalized_tuple_support,total_cases,mean_cases_per_leaf_tuple,national_share_pct
0,State (28),IT Act,504,43739,86.78,50.61
1,State (28),IPC,476,41539,87.27,48.07
2,State (28),SLL,140,325,2.32,0.38
3,Union Territory (8),IT Act,144,498,3.46,0.58
4,Union Territory (8),IPC,136,310,2.28,0.36
5,Union Territory (8),SLL,40,9,0.23,0.01


### AOI vs OLAP Comparison:
- **OLAP**: Focuses on multidimensional data cube navigation (slice/dice/drill-down/roll-up) for interactive user exploration.
- **AOI**: Focuses on automated data generalization by replacing specific attribute values with higher-level semantic concepts, creating compact prime relations for rule discovery.

## 7. Efficient Cube Computation & Iceberg Cuboids

In high-dimensional warehouses, materializing the full power set of cuboids ($2^d$) creates explosive storage and computational overhead. **Iceberg Cubes** selectively materialize only cells satisfying an analytical condition $\text{Measure} \ge T$.

We evaluate an Iceberg Cuboid on the Base State $\times$ Category table with threshold $T = 1,000$ cases.

In [10]:
iceberg_df = compute_iceberg_cuboid(threshold_cases=1000)
print(f"=== ICEBERG CUBOID (SUM(cases) >= 1,000): {len(iceberg_df)} Tuples Retained ===")
print(f"Total cases captured: {iceberg_df['cases'].sum():,} ({iceberg_df['cases'].sum()/86420*100:.2f}% of national total)")
display(iceberg_df)

=== ICEBERG CUBOID (SUM(cases) >= 1,000): 16 Tuples Retained ===
Total cases captured: 54,848 (63.47% of national total)


,state_name,is_ut,act_group,category_display_name,section_reference,cases,pct_of_national_total
0,Karnataka,0,IT Act,Computer Related Offences - D) Cheating by personation by using computer resource (Sec.66D),Sec.66D,18166,21.02
1,Telangana,0,IPC,Cheating (Sec.420 IPC),Sec.420 IPC,6384,7.39
2,Telangana,0,IPC,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - D) OTP Frauds","Sec.420 r/w Sec.465, 468- 471 IPC",4223,4.89
3,Maharashtra,0,IPC,Cheating (Sec.420 IPC),Sec.420 IPC,3743,4.33
4,Telangana,0,IPC,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - E) Others","Sec.420 r/w Sec.465, 468- 471 IPC",3100,3.59
5,Uttar Pradesh,0,IT Act,Computer Related Offences - Computer Related Offences (Sec.66) - a2) Offences other than Ransom-ware,Sec.66,2749,3.18
6,Uttar Pradesh,0,IT Act,Computer Related Offences - D) Cheating by personation by using computer resource (Sec.66D),Sec.66D,2748,3.18
7,Karnataka,0,IT Act,Computer Related Offences - C) Identity Theft (Sec.66C),Sec.66C,2411,2.79
8,Telangana,0,IPC,"Fraud (Sec.420 r/w Sec.465, 468- 471 IPC) - C) Online Banking Fraud","Sec.420 r/w Sec.465, 468- 471 IPC",1868,2.16
9,Kerala,0,IPC,Cheating (Sec.420 IPC),Sec.420 IPC,1719,1.99


### Iceberg Pruning Efficiency:
- **Full Base Cuboid**: $1,440$ tuples ($86,420$ cases).
- **Iceberg Cuboid ($T \ge 1,000$)**: **16 tuples** ($54,848$ cases).
- **Efficiency**: Retaining just **$1.11\%$ of tuples** captures **$63.47\%$ of national cybercrime volume**.

## 8. Visualizations & Pipeline Execution (Figures 33 to 35)

We execute the full Stage 11 pipeline to generate and export all analytical artifacts.

In [11]:
pipeline_outputs = run_stage11_pipeline()
print("=== GENERATED STAGE 11 ARTIFACTS ===")
for k, v in pipeline_outputs.items():
    p = Path(v)
    print(f"  - {k:<25}: {p.name} ({p.stat().st_size:,} bytes)")

=== EXECUTING STAGE 11: ADVANCED OLAP & DATA CUBE ANALYSIS ===
1. Computing multidimensional cuboids...
2. Generating State x Act Group Pivot table...
3. Performing Attribute-Oriented Induction (AOI)...
4. Computing Iceberg Cuboid (Threshold >= 1,000 cases)...
5. Generating Stage 11 visual artifacts...
=== STAGE 11 OLAP PIPELINE COMPLETED SUCCESSFULLY ===
=== GENERATED STAGE 11 ARTIFACTS ===
  - cube_state_category      : stage11_cube_state_category.csv (171,765 bytes)
  - cube_state_act_group     : stage11_cube_state_act_group.csv (4,001 bytes)
  - cube_national_category   : stage11_cube_national_category.csv (4,871 bytes)
  - cube_national_act_group  : stage11_cube_national_act_group.csv (123 bytes)
  - cube_admin_act_group     : stage11_cube_admin_act_group.csv (275 bytes)
  - cube_motive_summary      : stage11_cube_motive_summary.csv (693 bytes)
  - pivot_state_act_group    : stage11_pivot_state_act_group.csv (2,094 bytes)
  - aoi_generalized_relation : stage11_aoi_generalized_rela

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


## 9. Methodological Synthesis & Reconciliations

### Authoritative Reconciliations:
1. **National Total Cases**: Sum of all 40 leaf categories across 36 jurisdictions $= 86,420$ cases ($100.0\%$).
2. **Act Group Decomposition**: IT Act ($44,237$, $51.19\%$) + IPC ($41,849$, $48.43\%$) + SLL ($334$, $0.39\%$) $= 86,420$.
3. **Motive Reconciliation**: Sum of 18 specific motives across 36 jurisdictions $= 86,420$ motives ($100.0\%$).
4. **Pivot Total Reconciliations**: All row totals and column totals in `stage11_pivot_state_act_group.csv` match base facts with zero error.

### Small-N & Dimensional Limitations ($N = 36$):
- **Observational Unit**: Data represent aggregate State/UT jurisdictions, not individual crime incidents.
- **Separate Dimensional Spaces**: Categories and Motives cannot be cross-tabulated without raw incident microdata.
- **Descriptive Non-Causal Nature**: Cube roll-ups summarize observed reporting volumes without attributing causation.
